# R3 · English twins `_P` `_B` `_C` (Rathi)

**Run this only after the R2 notebook has finished and pushed.** This notebook needs R2's features and the reranker on Hugging Face Hub.

**Runtime → T4 GPU**, then **Run all** (about 1–1.5 h). It uses the same Colab Secrets as R2: `GITHUB_TOKEN`, `HF_TOKEN`, `WANDB_KEY`.

Steps: 1 setup → 2 10-row checks → 3 full twins → 4 checks + samples → 5 twin features → 6 push.

In [1]:
# 1. Setup. The repo is public, so cloning needs no token; GITHUB_TOKEN is only used to push at the end.
from google.colab import userdata
import os
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q rathi/R3-twins && git pull -q origin rathi/R3-twins
!git merge -q --no-edit origin/rathi/R2-reranker   # brings in features/ from the R2 run

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
ok = !git ls-remote https://{GH}@github.com/himaallu/Group-Project.git HEAD
print('GitHub token OK' if ok and 'fatal' not in ' '.join(ok) else 'GITHUB_TOKEN rejected: make a new one with the "repo" box ticked')

!pip install -q -U wandb bitsandbytes

import subprocess, sys
SPACY = 'en_core_web_trf'
if subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY]).returncode != 0 or \
        subprocess.run([sys.executable, '-c', f'import spacy; spacy.load("{SPACY}")']).returncode != 0:
    SPACY = 'en_core_web_lg'
    subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY], check=True)
print('spaCy model:', SPACY)

import wandb
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
wandb.login(key=userdata.get('WANDB_KEY').strip(), relogin=True)
from huggingface_hub import HfApi
RERANKER = f"{HfApi(token=os.environ['HF_TOKEN']).whoami()['name']}/twincheck-reranker-v1"
print('reranker:', RERANKER)
!ls -la features/

Cloning into '/content/twincheck'...
remote: Enumerating objects: 417, done.
remote: Counting objects: 100% (417/417), done.
remote: Compressing objects: 100% (230/230), done.
remote: Total 417 (delta 183), reused 339 (delta 127), pack-reused 0 (from 0)
Receiving objects: 100% (417/417), 34.21 MiB | 19.09 MiB/s, done.
Resolving deltas: 100% (183/183), done.
/content/twincheck
GitHub token OK
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
spaCy model: en_core_web_trf


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: rathiabeth (rathiabeth-uowd) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


reranker: RathiAbeth/twincheck-reranker-v1
total 49068
drwxr-xr-x  2 root root     4096 Nov 17 14:29 .
drwxr-xr-x 13 root root     4096 Nov 17 14:29 ..
-rw-r--r--  1 root root 49599376 Nov 17 14:29 entities_en.jsonl
-rw-r--r--  1 root root   633998 Nov 17 14:29 relevance_en.parquet


## 2. 10-row checks: 5 test + 5 train sources, all three twin types

In [2]:
!python -m pytest tests/copies -q
!python -m src.copies.make_twins --n-test 5 --n-train 5 --reranker {RERANKER} --spacy-model {SPACY}     --out-dir data/copies/en/test10 --report data/copies/en/test10/report.json
!python -m src.copies.checks --twins data/copies/en/test10/test.jsonl data/copies/en/test10/train.jsonl     --log data/copies/en/test10/twin_log.csv --out data/copies/en/test10/checks.json     --samples-dir data/copies/en/test10/samples
!head -c 1500 data/copies/en/test10/test.jsonl

.................                                                        [100%]
17 passed in 0.97s
name pools: 936 places, 2044 people
config.json: 100% 842/842 [00:00<00:00, 3.35MB/s]
tokenizer_config.json: 100% 409/409 [00:00<00:00, 1.82MB/s]

tokenizer.json: downloading bytes:   0% 0.00/17.1M [00:00<?, ?B/s]
tokenizer.json: downloading bytes:  20% 3.39M/17.1M [00:04<00:04, 3.13MB/s,  158kB/s  ]
tokenizer.json: downloading bytes: 100% 5.88M/5.88M [00:04<00:00, 1.27MB/s,  515kB/s  ]
tokenizer.json: reconstructing file: 100% 17.1M/17.1M [00:04<00:00, 3.68MB/s, 1.53MB/s  ]

model.safetensors: downloading bytes:   0% 0.00/1.14G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   7% 82.5M/1.14G [00:08<00:55, 19.1MB/s, 5.78MB/s  ]
model.safetensors: reconstructing file:   5% 60.5M/1.14G [00:08<02:17, 7.85MB/s, 3.20kB/s  ]
model.safetensors: reconstructing file:   5% 61.8M/1.14G [00:08<02:14, 7.96MB/s, 5.04MB/s  ]
model.safetensors: reconstructing file:   6% 63.6M/1.14G [00:09<02:47, 6.

## 3. Full run: 500 test + 3,000 train sources (about 1 h, mostly Qwen paraphrasing)

In [3]:
!python -m src.copies.make_twins --reranker {RERANKER} --spacy-model {SPACY}
print(open('results/R3/make_twins_report.json').read())

name pools: 936 places, 2044 people
Loading weights: 100% 393/393 [00:00<00:00, 6523.36it/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100% 399/399 [00:04<00:00, 88.89it/s] 
test: 500 sources (Counter({'multifc': 500}))
/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
  paraphrased 16/500
  paraphrased 32/500
  paraphrased 48/500
  paraphrased 64/500
  paraphrased 80/500
  paraphrased 96/500
  paraphrased 112/500
  paraphrased 128/500
  paraphrased 144/500
  paraphrased 160/500
  paraphrased 176/500
  paraphrased 192/500
  paraphrased 208/500
  paraphrased 224/500
  paraphrased 240/500
  paraphrased 256/500
  paraphra

## 4. Automatic checks + 10 examples of each twin type

In [4]:
!python -m src.copies.checks
import json
rep = json.load(open('results/R3/checks.json'))
for path, r in rep.items():
    print(path, r['per_type'], 'failed:', r['n_failed'], r['failures'][:3])

data/copies/en/test.jsonl: 955 twins {'P': 174, 'C': 500, 'B': 281}, 0 failed
data/copies/en/train.jsonl: 5705 twins {'B': 1719, 'C': 3000, 'P': 986}, 1 failed
data/copies/en/test.jsonl {'P': 174, 'C': 500, 'B': 281} failed: 0 []
data/copies/en/train.jsonl {'B': 1719, 'C': 3000, 'P': 986} failed: 1 [{'id': 'multifc_030465_C', 'problems': ['_C donor relevance >= 0.1']}]


## 5. Relevance and entities for the twin snippets (Himasri's verdict head and R4 need these)

In [5]:
!python -m src.retrieval.rerank data/copies/en/test.jsonl data/copies/en/train.jsonl     --model {RERANKER} --out features/relevance_copies_en.parquet
!python -m src.data.entities data/copies/en/test.jsonl data/copies/en/train.jsonl     --out features/entities_copies_en.jsonl --model {SPACY}
!du -h data/copies/en/* features/*copies*

Loading weights: 100% 393/393 [00:00<00:00, 5205.40it/s]
data/copies/en/test.jsonl: 955 records, 6647 snippets (14s on cuda)
data/copies/en/train.jsonl: 5705 records, 42266 snippets (91s on cuda)
48913 rows -> features/relevance_copies_en.parquet; mean relevance 0.477
/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/thinc/util.py:401: VisibleDeprecationWarning: This function is deprecated and will be removed in a future release. Use the cupy.from_dlpack() array constructor instead.
  dlpack_tensor = xp_tensor.toDlpack()  # type: ignore
data/copies/en/test.jsonl: 955 records (36s)
data/copies/en/train.jsonl: 5705 records (187s)
6660 records -> features/entities_copies_en.jsonl
112K	data/copies/en/sources.csv
4.1M	data/copies/en/staging
24K	data/copies/en/test10/staging
80K	data/copies/en/test10/samples
200K	data/c

## 6. Push to `rathi/R3-twins`

In [6]:
!git add data/copies/en/test.jsonl data/copies/en/train.jsonl data/copies/en/sources.csv data/copies/en/twin_log.csv     features/relevance_copies_en.parquet features/entities_copies_en.jsonl     results/R3/make_twins_report.json results/R3/checks.json results/R3/samples
!git commit -q -m "[R3] Add English twins (_P _B _C), sources, checks and twin features from Colab run"
!git push -q origin rathi/R3-twins && echo PUSHED

PUSHED


**Done.** Tell Claude it finished, or paste the output of steps 3–4. Then **Runtime → Disconnect and delete runtime**.